In [1]:
# Cell 1: Setup and configuration
import json
import pandas as pd
import time
import random
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Load API key from .env file
load_dotenv(r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\.env")

client = Anthropic()

# Our concern categories and urgency bands
CATEGORIES = [
    "Bullying / peer conflict",
    "Mental health / self-harm",
    "Home issues / neglect",
    "Online safety",
    "Physical safety",
    "Abuse by adult in organisation",
    "Attendance / engagement",
    "Exploitation / trafficking",
    "Radicalisation / extremism",
    "Financial hardship",
    "FGM / harmful practices",
    "Behaviour / conduct",
]

URGENCY_BANDS = ["Low", "Medium", "High", "Critical"]

REPORTER_ROLES = ["CFAV", "Cadet", "Parent/Carer", "Civilian Instructor", "Commanding Officer"]
SETTINGS = ["Parade Night", "Weekend Exercise", "Annual Camp", "Online/Remote", "Off-site Activity"]
AGE_BANDS = ["Under 12", "12-14", "14-16", "16-18"]

print("Setup complete.")
print(f"Categories: {len(CATEGORIES)}")
print(f"Urgency bands: {URGENCY_BANDS}")

Setup complete.
Categories: 12
Urgency bands: ['Low', 'Medium', 'High', 'Critical']


In [2]:
# Cell 2: Generation prompt and function

SYSTEM_PROMPT = """You are an expert in youth safeguarding within the UK Cadet Forces (Army Cadets, Sea Cadets, Air Cadets, Combined Cadet Force). You have extensive experience writing and reviewing safeguarding concern logs.

Your task is to generate realistic synthetic safeguarding concern narratives for training an NLP model. Each narrative should read as if written by a real person recording a genuine concern.

CRITICAL RULES:
1. Write ONLY what was observed, disclosed or reported. Do NOT include any action taken, response planned, or follow-up language.
2. Do NOT include any phrases that signal the urgency level such as "low-level", "significant concern", "immediate risk", "emergency", "monitor", "escalate", "refer" or similar.
3. The urgency must come from WHAT IS DESCRIBED, not from how the writer characterises it.
4. Vary the writing style - some reporters write formally, others informally. Some give lots of detail, others are brief.
5. Never repeat the same scenario. Every narrative must describe a genuinely different situation.
6. Use realistic UK English language and terminology appropriate to the Cadet Forces.

URGENCY GUIDELINES (do not state these in the narrative):
- Low: Minor welfare observations. A cadet seems a bit quiet, mild friendship issues, minor hygiene concerns, occasional lateness.
- Medium: A specific concern needing attention. Persistent bullying, noticeable changes in behaviour over weeks, disclosed family arguments, low mood affecting participation.
- High: Clear safeguarding issue. Disclosed self-harm, visible unexplained injuries, concerning online contact with unknown adults, disclosure of domestic violence at home, substance misuse.
- Critical: Immediate risk of serious harm. Disclosure of sexual abuse, physical abuse by adult in position of trust, active suicidal intent, disclosure of exploitation, evidence of significant physical harm."""

def generate_batch(category, urgency, count=5):
    """Generate a batch of synthetic safeguarding concern narratives."""
    
    prompt = f"""Generate exactly {count} safeguarding concern narratives for the following:
- Concern category: {category}
- Urgency band: {urgency}

Return ONLY a JSON array of strings, where each string is one narrative. Example format:
["Narrative one text here.", "Narrative two text here."]

Each narrative should be 2-4 sentences long (30-70 words). Remember: describe ONLY what was observed or disclosed. No action language, no urgency signalling phrases."""

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    
    raw_text = response.content[0].text
    
    try:
        narratives = json.loads(raw_text)
        return narratives
    except json.JSONDecodeError:
        # Try to extract JSON array from the response
        start = raw_text.find("[")
        end = raw_text.rfind("]") + 1
        if start >= 0 and end > start:
            narratives = json.loads(raw_text[start:end])
            return narratives
        else:
            print(f"  WARNING: Could not parse response for {category}/{urgency}")
            return []

# Test with a single small batch
test_narratives = generate_batch("Bullying / peer conflict", "Medium", count=3)
print(f"Generated {len(test_narratives)} test narratives:\n")
for i, text in enumerate(test_narratives):
    print(f"{i+1}. {text}\n")

Generated 3 test narratives:

1. During the evening meal, I observed Cadet Smith deliberately knocking Cadet Jones's food tray to the floor while walking past. When Jones bent to clean it up, Smith laughed and said 'clumsy idiot' loud enough for others to hear. This is the third incident I've witnessed this week where Smith has targeted Jones specifically.

2. Cadet Williams approached me after drill to say that a group of older cadets have been sending her nasty messages on social media for the past two weeks. She showed me screenshots of messages calling her 'pathetic' and telling her she doesn't belong in cadets. Williams said it's been happening since she joined the unit last month.

3. I noticed Cadet Brown sitting alone again during the break, looking upset. When I asked if he was alright, he told me that some cadets have been hiding his kit and calling him names because of his accent. He said it's been going on for about three weeks and he's started dreading coming to cadets.



In [9]:
# Cell 3: Generate full synthetic dataset
all_records = []
record_id = 1

# We want roughly 1200 records: 12 categories × 4 urgency bands × ~25 narratives each
NARRATIVES_PER_COMBINATION = 25
BATCH_SIZE = 5  # Generate 5 narratives per API call

total_combinations = len(CATEGORIES) * len(URGENCY_BANDS)
print(f"Generating {NARRATIVES_PER_COMBINATION} narratives for each of {total_combinations} category/urgency combinations")
print(f"Target total: {total_combinations * NARRATIVES_PER_COMBINATION} records")
print(f"API calls needed: {total_combinations * (NARRATIVES_PER_COMBINATION // BATCH_SIZE)}")
print()

for cat in CATEGORIES:
    for urg in URGENCY_BANDS:
        print(f"Generating: {cat} / {urg} ...", end=" ")
        narratives_for_combo = []
        
        batches_needed = NARRATIVES_PER_COMBINATION // BATCH_SIZE
        for batch_num in range(batches_needed):
            try:
                batch = generate_batch(cat, urg, count=BATCH_SIZE)
                narratives_for_combo.extend(batch)
                time.sleep(1)  # Brief pause to avoid rate limits
            except Exception as e:
                print(f"\n  ERROR on batch {batch_num+1}: {e}")
                time.sleep(5)
        
        # Build records with metadata
        for narrative in narratives_for_combo:
            record = {
                "report_id": record_id,
                "reporter_role": random.choice(REPORTER_ROLES),
                "setting": random.choice(SETTINGS),
                "age_band": random.choice(AGE_BANDS),
                "category_label": cat,
                "urgency_label": urg,
                "free_text": narrative,
            }
            all_records.append(record)
            record_id += 1
        
        print(f"got {len(narratives_for_combo)} narratives")

df_new = pd.DataFrame(all_records)
print(f"\nTotal records generated: {len(df_new)}")
print(f"\nUrgency distribution:")
print(df_new["urgency_label"].value_counts().sort_index())
print(f"\nCategory distribution:")
print(df_new["category_label"].value_counts().sort_index())

Generating 25 narratives for each of 48 category/urgency combinations
Target total: 1200 records
API calls needed: 240

Generating: Bullying / peer conflict / Low ... got 25 narratives
Generating: Bullying / peer conflict / Medium ... got 25 narratives
Generating: Bullying / peer conflict / High ... got 25 narratives
Generating: Bullying / peer conflict / Critical ... got 25 narratives
Generating: Mental health / self-harm / Low ... got 25 narratives
Generating: Mental health / self-harm / Medium ... got 25 narratives
Generating: Mental health / self-harm / High ... got 25 narratives
Generating: Mental health / self-harm / Critical ... got 25 narratives
Generating: Home issues / neglect / Low ... got 25 narratives
Generating: Home issues / neglect / Medium ... got 25 narratives
Generating: Home issues / neglect / High ... got 25 narratives
Generating: Home issues / neglect / Critical ... got 25 narratives
Generating: Online safety / Low ... got 25 narratives
Generating: Online safety /

In [10]:
# Cell 4: Save the new dataset
output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v2.csv"
df_new.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Quick quality check - show one random example from each urgency band
print("\n=== SAMPLE NARRATIVES ===\n")
for urg in URGENCY_BANDS:
    sample = df_new[df_new["urgency_label"] == urg].sample(1, random_state=42).iloc[0]
    print(f"[{urg.upper()}] Category: {sample['category_label']}")
    print(f"  {sample['free_text']}")
    print()

Dataset saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v2.csv
File size: 395.7 KB

=== SAMPLE NARRATIVES ===

[LOW] Category: Radicalisation / extremism
  Cadet Brown brought up some political views during break time that sounded quite mature for his age - talking about immigration policy and national identity. The language seemed like it might have come from adult conversations or online content rather than his own thinking.

[MEDIUM] Category: Radicalisation / extremism
  Cadet Harris disclosed that he's been attending meetings with a group that his uncle introduced him to. He described them as 'real patriots who understand what needs to be done' and mentioned they've been teaching him about 'the threat to traditional British culture' from various minority communities.

[HIGH] Category: Radicalisation / extremism
  I overheard Cadet Martinez telling another cadet about joining online groups where they share 'the truth about

In [13]:
# Cell 5: Generate data for new categories
NEW_CATEGORIES = [
    "Sexual abuse / assault",
    "Grooming",
    "Abuse by another young person",
]

new_records = []
record_id = 2000  # Start from a high number to avoid clashing with existing IDs

for cat in NEW_CATEGORIES:
    for urg in URGENCY_BANDS:
        print(f"Generating: {cat} / {urg} ...", end=" ")
        narratives_for_combo = []

        batches_needed = NARRATIVES_PER_COMBINATION // BATCH_SIZE  # 25 // 5 = 5 batches
        for batch_num in range(batches_needed):
            try:
                batch = generate_batch(cat, urg, count=BATCH_SIZE)
                narratives_for_combo.extend(batch)
                time.sleep(1)
            except Exception as e:
                print(f"\n  ERROR on batch {batch_num+1}: {e}")
                time.sleep(5)

        for narrative in narratives_for_combo:
            record = {
                "report_id": record_id,
                "reporter_role": random.choice(REPORTER_ROLES),
                "setting": random.choice(SETTINGS),
                "age_band": random.choice(AGE_BANDS),
                "category_label": cat,
                "urgency_label": urg,
                "free_text": narrative,
            }
            new_records.append(record)
            record_id += 1

        print(f"got {len(narratives_for_combo)} narratives")

df_new_cats = pd.DataFrame(new_records)
print(f"\nNew category records generated: {len(df_new_cats)}")
print(f"\nCategory distribution:")
print(df_new_cats["category_label"].value_counts())
print(f"\nUrgency distribution:")
print(df_new_cats["urgency_label"].value_counts().sort_index())

Generating: Sexual abuse / assault / Low ... got 25 narratives
Generating: Sexual abuse / assault / Medium ... got 25 narratives
Generating: Sexual abuse / assault / High ... got 25 narratives
Generating: Sexual abuse / assault / Critical ... got 25 narratives
Generating: Grooming / Low ... got 25 narratives
Generating: Grooming / Medium ... got 25 narratives
Generating: Grooming / High ... got 25 narratives
Generating: Grooming / Critical ... got 25 narratives
Generating: Abuse by another young person / Low ... got 25 narratives
Generating: Abuse by another young person / Medium ... got 25 narratives
Generating: Abuse by another young person / High ... got 25 narratives
Generating: Abuse by another young person / Critical ... got 25 narratives

New category records generated: 300

Category distribution:
category_label
Sexual abuse / assault           100
Grooming                         100
Abuse by another young person    100
Name: count, dtype: int64

Urgency distribution:
urgency_l

In [14]:
# Cell 6: Combine with existing v2 data and save
df_v2 = pd.read_csv(r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v2.csv")
print(f"Existing v2 records: {len(df_v2)}")
print(f"New category records: {len(df_new_cats)}")

df_v3 = pd.concat([df_v2, df_new_cats], ignore_index=True)
print(f"Combined v3 records: {len(df_v3)}")

# Save
output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v3.csv"
df_v3.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

print(f"\nFull category distribution:")
print(df_v3["category_label"].value_counts().sort_index())
print(f"\nUrgency distribution:")
print(df_v3["urgency_label"].value_counts().sort_index())

Existing v2 records: 1200
New category records: 300
Combined v3 records: 1500

Saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v3.csv

Full category distribution:
category_label
Abuse by adult in organisation    100
Abuse by another young person     100
Attendance / engagement           100
Behaviour / conduct               100
Bullying / peer conflict          100
Exploitation / trafficking        100
FGM / harmful practices           100
Financial hardship                100
Grooming                          100
Home issues / neglect             100
Mental health / self-harm         100
Online safety                     100
Physical safety                   100
Radicalisation / extremism        100
Sexual abuse / assault            100
Name: count, dtype: int64

Urgency distribution:
urgency_label
Critical    375
High        375
Low         375
Medium      375
Name: count, dtype: int64


In [4]:
# Cell 7: Generate targeted data for weak categories

BATCH_SIZE = 5

WEAK_CATEGORIES = {
    "Behaviour / conduct": 40,
    "Grooming": 40,
    "Physical safety": 40,
    "Bullying / peer conflict": 40,
    "Abuse by another young person": 40,
    "Home issues / neglect": 35,
    "Online safety": 35,
    "Mental health / self-harm": 35,
    "Exploitation / trafficking": 35,
    "Sexual abuse / assault": 35,
    "Attendance / engagement": 25,
    "Abuse by adult in organisation": 25,
    "Financial hardship": 25,
    "Radicalisation / extremism": 25,
    "FGM / harmful practices": 25,
}

total_target = sum(WEAK_CATEGORIES.values())
print(f"Target: {total_target} new records across {len(WEAK_CATEGORIES)} categories")
print(f"API calls needed: {sum(count // BATCH_SIZE * len(URGENCY_BANDS) for count in WEAK_CATEGORIES.values())}")
print()

extra_records = []
record_id = 5000

for cat, count_per_urgency in WEAK_CATEGORIES.items():
    for urg in URGENCY_BANDS:
        narratives_for_combo = []
        batches_needed = count_per_urgency // BATCH_SIZE

        print(f"Generating: {cat} / {urg} ({count_per_urgency} records) ...", end=" ")

        for batch_num in range(batches_needed):
            try:
                batch = generate_batch(cat, urg, count=BATCH_SIZE)
                narratives_for_combo.extend(batch)
                time.sleep(1)
            except Exception as e:
                print(f"\n  ERROR on batch {batch_num+1}: {e}")
                time.sleep(5)

        for narrative in narratives_for_combo:
            record = {
                "report_id": record_id,
                "reporter_role": random.choice(REPORTER_ROLES),
                "setting": random.choice(SETTINGS),
                "age_band": random.choice(AGE_BANDS),
                "category_label": cat,
                "urgency_label": urg,
                "free_text": narrative,
            }
            extra_records.append(record)
            record_id += 1

        print(f"got {len(narratives_for_combo)}")

df_extra = pd.DataFrame(extra_records)
print(f"\nTotal extra records: {len(df_extra)}")
print(f"\nCategory distribution:")
print(df_extra["category_label"].value_counts().sort_index())
print(f"\nUrgency distribution:")
print(df_extra["urgency_label"].value_counts().sort_index())

Target: 500 new records across 15 categories
API calls needed: 400

Generating: Behaviour / conduct / Low (40 records) ... got 40
Generating: Behaviour / conduct / Medium (40 records) ... got 40
Generating: Behaviour / conduct / High (40 records) ... got 40
Generating: Behaviour / conduct / Critical (40 records) ... got 40
Generating: Grooming / Low (40 records) ... got 40
Generating: Grooming / Medium (40 records) ... got 40
Generating: Grooming / High (40 records) ... got 40
Generating: Grooming / Critical (40 records) ... got 40
Generating: Physical safety / Low (40 records) ... got 40
Generating: Physical safety / Medium (40 records) ... got 40
Generating: Physical safety / High (40 records) ... got 40
Generating: Physical safety / Critical (40 records) ... got 40
Generating: Bullying / peer conflict / Low (40 records) ... got 40
Generating: Bullying / peer conflict / Medium (40 records) ... got 40
Generating: Bullying / peer conflict / High (40 records) ... got 40
Generating: Bull

In [5]:
# Cell 8: Combine with existing v3 data and save as v4
df_v3 = pd.read_csv(r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v3.csv")
print(f"Existing v3 records: {len(df_v3)}")
print(f"New extra records: {len(df_extra)}")

df_v4 = pd.concat([df_v3, df_extra], ignore_index=True)
print(f"Combined v4 records: {len(df_v4)}")

output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v4.csv"
df_v4.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

print(f"\nCategory distribution:")
print(df_v4["category_label"].value_counts().sort_index())
print(f"\nUrgency distribution:")
print(df_v4["urgency_label"].value_counts().sort_index())

Existing v3 records: 1500
New extra records: 2000
Combined v4 records: 3500

Saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v4.csv

Category distribution:
category_label
Abuse by adult in organisation    200
Abuse by another young person     260
Attendance / engagement           200
Behaviour / conduct               260
Bullying / peer conflict          260
Exploitation / trafficking        240
FGM / harmful practices           200
Financial hardship                200
Grooming                          260
Home issues / neglect             240
Mental health / self-harm         240
Online safety                     240
Physical safety                   260
Radicalisation / extremism        200
Sexual abuse / assault            240
Name: count, dtype: int64

Urgency distribution:
urgency_label
Critical    875
High        875
Low         875
Medium      875
Name: count, dtype: int64
